In [1]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import yfinance as yf
from plotly.subplots import make_subplots

# Import formulas from formulas.py
import sys
sys.path.append('/Users/henrikengdal/Documents/GitHub/IND310-Project')
import formulas

# Define tickers and date range (same as Overview.py)
TICKERS = ["EQNR.OL", "DNB.OL", "AKRBP.OL", "ORK.OL", "MOWI.OL"]
START = "2020-10-15"
END = "2025-10-15"

# Download data for all tickers (same approach as data_cache.py)
print("Downloading stock data...")
all_data = {}
for ticker in TICKERS:
    print(f"Downloading {ticker}...")
    ticker_data = yf.Ticker(ticker).history(start=START, end=END, actions=True, auto_adjust=False)
    if not ticker_data.empty:
        ticker_data.index = ticker_data.index.tz_localize(None)
        all_data[ticker] = ticker_data

# Download market index for beta calculation
print("Downloading market index (OSEBX.OL)...")
market_data = yf.Ticker("OSEBX.OL").history(start=START, end=END)
if not market_data.empty:
    market_data.index = market_data.index.tz_localize(None)
    market_returns = market_data['Close'].pct_change().dropna()
else:
    market_returns = None
    print("Warning: Could not download market data")

# Combine data into MultiIndex DataFrame
data = pd.concat(all_data, axis=1)

# Extract close, dividends, and splits
close_data = data.xs('Close', level=1, axis=1)
dividends_data = data.xs('Dividends', level=1, axis=1)
splits_data = data.xs('Stock Splits', level=1, axis=1)

# Calculate adjusted close using formulas.py
print("Calculating adjusted close prices...")
adj_close_data = formulas.adjusted_close_price(close_data, dividends_data, splits_data)

# Calculate returns using formulas.py
print("Calculating returns...")
returns = formulas.returns(adj_close_data)

# Calculate beta for each ticker
print("Calculating beta values...")
betas = {}
if market_returns is not None:
    for ticker in TICKERS:
        if ticker in returns.columns:
            betas[ticker] = formulas.beta(returns[ticker], market_returns)
            print(f"{ticker} Beta: {betas[ticker]:.4f}")
else:
    for ticker in TICKERS:
        betas[ticker] = 0.0

print(f"\nData shape: {adj_close_data.shape}")
print("\nAdjusted close prices (first 5 rows):")
adj_close_data.head()

Calculating adjusted close prices...
Calculating returns...
Calculating beta values...
EQNR.OL Beta: 1.4029
DNB.OL Beta: 0.8574
AKRBP.OL Beta: 1.5490
ORK.OL Beta: 0.2157
MOWI.OL Beta: 0.8276

Data shape: (1257, 5)

Adjusted close prices (first 5 rows):
Calculating adjusted close prices...
Calculating returns...
Calculating beta values...
EQNR.OL Beta: 1.4029
DNB.OL Beta: 0.8574
AKRBP.OL Beta: 1.5490
ORK.OL Beta: 0.2157
MOWI.OL Beta: 0.8276

Data shape: (1257, 5)

Adjusted close prices (first 5 rows):


,EQNR.OL,DNB.OL,AKRBP.OL,ORK.OL,MOWI.OL
Date,,,,,
2020-10-15,132.199997,132.750000,149.050003,92.080002,168.250000
2020-10-16,133.850006,134.750000,152.000000,92.940002,168.250000
2020-10-19,133.550003,138.300003,152.000000,92.860001,170.350006
2020-10-20,132.100006,138.500000,150.000000,92.900002,169.899994
2020-10-21,131.600006,140.000000,150.000000,92.199997,164.699997


In [3]:
# Define consistent colors (same as Overview.py)
color_map = {
    "EQNR.OL": "#2f4d8e",
    "DNB.OL": "#ffa659",
    "AKRBP.OL": "#ff6464",
    "ORK.OL": "#79ffbc",
    "MOWI.OL": "#c180ff",
}
colors = [color_map[ticker] for ticker in TICKERS]

# Plot 1: Adjusted Close Prices
fig = go.Figure()

for i, ticker in enumerate(TICKERS):
    if ticker in adj_close_data.columns:
        fig.add_trace(go.Scatter(
            x=adj_close_data.index,
            y=adj_close_data[ticker],
            mode='lines',
            name=ticker,
            line=dict(color=colors[i], width=2),
            hovertemplate=f'<b>{ticker}</b><br>' +
                         'Dato: %{x}<br>' +
                         'Justert sluttkurs: %{y:.2f} NOK<br>' +
                         '<extra></extra>'
        ))

fig.update_layout(
    title='Historisk justert sluttkurs - Norske Selskaper',
    xaxis_title='Dato',
    yaxis_title='Justert sluttkurs (NOK)',
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="black",
        borderwidth=1
    ),
    height=600,
    width=1000,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig.show()

In [4]:
# Plot 2: Risk Metrics (Standard Deviation)
risk_metrics = []
for ticker in TICKERS:
    if ticker in returns.columns:
        daily_std = returns[ticker].std()
        annual_std = daily_std * (252**0.5)
        risk_metrics.append({
            'Ticker': ticker,
            'Daily_Std': daily_std * 100,
            'Annual_Std': annual_std * 100
        })

risk_df = pd.DataFrame(risk_metrics)

# Create subplot with standard deviation charts
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Daglig standardavvik (%)', 'Årlig standardavvik (%)'),
    horizontal_spacing=0.1
)

fig.add_trace(
    go.Bar(
        x=risk_df['Ticker'],
        y=risk_df['Daily_Std'],
        name='Daglig std',
        marker_color=colors,
        text=[f'{val:.4f}%' for val in risk_df['Daily_Std']],
        textposition='outside',
        textfont=dict(size=10, color='black')
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=risk_df['Ticker'],
        y=risk_df['Annual_Std'],
        name='Årlig std',
        marker_color=colors,
        text=[f'{val:.2f}%' for val in risk_df['Annual_Std']],
        textposition='outside',
        textfont=dict(size=10, color='black')
    ),
    row=1, col=2
)

fig.update_layout(
    title='Risikoanalyse: Standardavvik for norske aksjer',
    height=550,
    width=1000,
    showlegend=False,
    plot_bgcolor='white'
)

fig.update_yaxes(title_text="Standardavvik (%)", row=1, col=1, showgrid=True, gridcolor='lightgray')
fig.update_yaxes(title_text="Standardavvik (%)", row=1, col=2, showgrid=True, gridcolor='lightgray')

fig.show()

In [ ]:
# Plot 3: Beta Values
beta_df = pd.DataFrame([
    {'Ticker': ticker, 'Beta': betas[ticker]}
    for ticker in TICKERS
])

fig = go.Figure()

# Bar chart for beta
fig.add_trace(go.Bar(
    x=beta_df['Ticker'],
    y=beta_df['Beta'],
    marker_color=colors,
    text=[f'{val:.3f}' for val in beta_df['Beta']],
    textposition='outside',
    textfont=dict(size=12, color='black'),
    hovertemplate='<b>%{x}</b><br>Beta: %{y:.3f}<extra></extra>'
))

# Add reference line at beta = 1
fig.add_hline(
    y=1, 
    line_dash="dash", 
    line_color="gray",
    annotation_text="Marked",
    annotation_position="right"
)

fig.update_layout(
    title='Beta verdier mot OSEBX',
    xaxis_title='',
    yaxis_title='Beta (β)',
    height=500,
    width=1000,
    plot_bgcolor='white',
    yaxis=dict(
        showgrid=True, 
        gridcolor='lightgray',
        range=[0, max(beta_df['Beta'].max() * 1.2, 1.5)]
    ),
    xaxis=dict(showgrid=False)
)

fig.show()

# Print beta interpretation
print("\n" + "="*60)
print("BETA ANALYSE (mot OSEBX)")
print("="*60)
print("Beta måler korrelasjonen med markedet:")
print("  β > 1: Mer volatil enn markedet")
print("  β = 1: Beveger seg med markedet")
print("  β < 1: Mindre volatil enn markedet")
print("\n" + "-"*60)

for _, row in beta_df.iterrows():
    interpretation = "Høyere volatilitet" if row['Beta'] > 1 else "Lavere volatilitet" if row['Beta'] < 1 else "Lik markedet"
    print(f"{row['Ticker']:10} | Beta: {row['Beta']:6.3f} | {interpretation}")


BETA ANALYSE (mot OSEBX)
Beta måler korrelasjonen med markedet:
  β > 1: Mer volatil enn markedet
  β = 1: Beveger seg med markedet
  β < 1: Mindre volatil enn markedet

------------------------------------------------------------
EQNR.OL    | Beta:  1.403 | Høyere volatilitet
DNB.OL     | Beta:  0.857 | Lavere volatilitet
AKRBP.OL   | Beta:  1.549 | Høyere volatilitet
ORK.OL     | Beta:  0.216 | Lavere volatilitet
MOWI.OL    | Beta:  0.828 | Lavere volatilitet


In [6]:
# Combined Risk and Beta Analysis
print("\n" + "="*60)
print("KOMPLETT RISIKOANALYSE")
print("="*60)

combined_metrics = risk_df.merge(beta_df, on='Ticker')
combined_metrics = combined_metrics.sort_values('Beta', ascending=False)

print(f"\n{'Ticker':<10} {'Beta':>8} {'Daglig Std':>12} {'Årlig Std':>12}")
print("-"*50)
for _, row in combined_metrics.iterrows():
    print(f"{row['Ticker']:<10} {row['Beta']:>8.3f} {row['Daily_Std']:>11.4f}% {row['Annual_Std']:>11.2f}%")

# Correlation matrix
print(f"\n{'='*60}")
print("KORRELASJONSMATRISE (Returns)")
print("="*60)
correlation_matrix = returns.corr()
print(correlation_matrix.round(3))


KOMPLETT RISIKOANALYSE

Ticker         Beta   Daglig Std    Årlig Std
--------------------------------------------------
AKRBP.OL      1.549      2.1788%       34.59%
EQNR.OL       1.403      1.9451%       30.88%
DNB.OL        0.857      1.4622%       23.21%
MOWI.OL       0.828      1.6373%       25.99%
ORK.OL        0.216      1.2221%       19.40%

KORRELASJONSMATRISE (Returns)
          EQNR.OL  DNB.OL  AKRBP.OL  ORK.OL  MOWI.OL
EQNR.OL     1.000   0.241     0.739   0.004    0.146
DNB.OL      0.241   1.000     0.259   0.102    0.296
AKRBP.OL    0.739   0.259     1.000  -0.030    0.170
ORK.OL      0.004   0.102    -0.030   1.000    0.186
MOWI.OL     0.146   0.296     0.170   0.186    1.000


In [7]:
# Create comparison chart: Regular Close vs Adjusted Close
from plotly.subplots import make_subplots

# Select one ticker for detailed comparison (you can change this)
sample_ticker = "EQNR.OL"

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(f'{sample_ticker} - Regular Close Price', f'{sample_ticker} - Adjusted Close Price'),
    vertical_spacing=0.1
)

# Regular close price
fig.add_trace(
    go.Scatter(
        x=close_prices.index,
        y=close_prices[sample_ticker],
        mode='lines',
        name='Regular Close',
        line=dict(color='blue', width=2)
    ),
    row=1, col=1
)

# Adjusted close price
fig.add_trace(
    go.Scatter(
        x=adjusted_close_prices.index,
        y=adjusted_close_prices[sample_ticker],
        mode='lines',
        name='Adjusted Close',
        line=dict(color='red', width=2)
    ),
    row=2, col=1
)

fig.update_layout(
    title=f'Sammenligning: Vanlig vs Justert Sluttkurs - {sample_ticker}',
    height=800,
    showlegend=True
)

fig.update_xaxes(title_text="Dato", row=2, col=1)
fig.update_yaxes(title_text="Pris (NOK)", row=1, col=1)
fig.update_yaxes(title_text="Justert Pris (NOK)", row=2, col=1)

fig.show()

# Show the difference in actual numbers
print(f"\n{sample_ticker} - Price Comparison (Last 10 days):")
comparison_df = pd.DataFrame({
    'Date': close_prices.index[-10:],
    'Regular Close': close_prices[sample_ticker].iloc[-10:].values,
    'Adjusted Close': adjusted_close_prices[sample_ticker].iloc[-10:].values
})
comparison_df['Difference'] = comparison_df['Regular Close'] - comparison_df['Adjusted Close']
comparison_df['Difference %'] = ((comparison_df['Regular Close'] - comparison_df['Adjusted Close']) / comparison_df['Regular Close'] * 100).round(4)

print(comparison_df.to_string(index=False))

NameError: name 'close_prices' is not defined